# Regularization for Deep Learning — Notebook 1 of 3
## Parameter Norm Penalties · L² Weight Decay · The Hessian View

**Companion to Chapter 7, Stations 1–3.**  This is the first of three notebooks. It builds the core intuition every later idea rests on:

> A norm penalty is a **second force** on the weights. The data term pulls the weights toward the fit `w*`; the penalty pulls them toward `0`; the solution `w̃` is the balance point.

### What you will do here
1. **See overfitting** — the problem regularization exists to solve.
2. **Weight decay** — watch weights shrink by a factor `(1 − εα)` every step.
3. **A real training loop** — flip `weight_decay` on and off in PyTorch.
4. **The Hessian view** — `w̃ = (H + αI)⁻¹ H w*`, and why steep directions survive while flat ones decay.
5. **Ridge regression** — the same story written in the language of data.

Run every cell top to bottom (`Runtime → Run all`). Cells marked **🔧 Try it** are for you to edit. Cells marked **✏️ Exercise** ask you to write a little code.

*Nothing here needs a GPU.*

## 0 · Setup
Run this once. Everything below uses only NumPy, Matplotlib, scikit-learn and (optionally) PyTorch — all pre-installed in Colab.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)
plt.rcParams["figure.figsize"] = (7, 4.2)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

# A consistent colour language across all three notebooks:
C_DATA = "#0F7C87"   # teal  — the data term / w*
C_L2   = "#6C30D9"   # violet— L2 / weight decay
C_L1   = "#C81C6E"   # rose  — L1 / sparsity
C_WARN = "#B27412"   # amber — caution
print("Ready. numpy", np.__version__)

## 1 · The equation, and the problem it solves

The regularized objective is just the ordinary loss plus a penalty on the size of the weights:

$$\tilde J(\theta;X,y) \;=\; \underbrace{J(\theta;X,y)}_{\text{data term}} \;+\; \alpha\,\underbrace{\Omega(\theta)}_{\text{penalty}},\qquad \alpha \in [0,\infty)$$

* **Data term** `J` — the usual loss (MSE, cross-entropy). Its gradient pulls weights toward `w*`, the best training fit.
* **Penalty** `αΩ` — a spring back toward the origin. Its gradient pulls weights toward `0`.
* **α is the dial** — `α = 0` is the unregularized problem; bigger `α` shrinks the weights harder. It is *tuned on validation*, not learned.
* We penalize the **weights `w`**, not the **biases `b`** — a bias controls one variable and is fit accurately from little data, so penalizing it mostly adds underfitting for no gain.

Let's *see* the overfitting that motivates all of this.

In [ ]:
# A tiny 1-D regression: the truth is a gentle curve, the data is noisy.
def true_fn(x):
    return np.sin(1.3 * x) + 0.3 * x

n_train = 18
x_train = np.sort(np.random.uniform(-3, 3, n_train))
y_train = true_fn(x_train) + np.random.normal(0, 0.35, n_train)

x_grid = np.linspace(-3.2, 3.2, 400)

# Fit a HIGH-degree polynomial with plain least squares (no regularization).
degree = 12
def poly_features(x, d):
    return np.vstack([x**k for k in range(d + 1)]).T

Xtr = poly_features(x_train, degree)
w_ols, *_ = np.linalg.lstsq(Xtr, y_train, rcond=None)
y_fit = poly_features(x_grid, degree) @ w_ols

plt.figure()
plt.plot(x_grid, true_fn(x_grid), "--", color="gray", label="true function")
plt.scatter(x_train, y_train, color=C_DATA, zorder=5, label="training data")
plt.plot(x_grid, y_fit, color=C_L1, lw=2, label=f"degree-{degree} least squares")
plt.ylim(-3, 3.5); plt.legend(); plt.title("Unregularized: the curve wiggles to hit every point")
plt.show()

print("Largest weight magnitude:", np.max(np.abs(w_ols)).round(1),
      "  <-- huge weights are the fingerprint of overfitting")

### The train–validation gap

Overfitting shows up as a **gap**: near-perfect on training data, poor on data it has not seen. Let's measure that gap as we turn `α` up using **ridge regression** (L² on a linear model — Station 3). Watch the validation error fall then rise: too little `α` overfits, too much underfits.

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error

# Held-out validation set drawn from the same truth
x_val = np.sort(np.random.uniform(-3, 3, 200))
y_val = true_fn(x_val) + np.random.normal(0, 0.35, 200)
Xval = poly_features(x_val, degree)

alphas = np.logspace(-4, 3, 40)
train_err, val_err = [], []
for a in alphas:
    model = Ridge(alpha=a, fit_intercept=False).fit(Xtr, y_train)
    train_err.append(mean_squared_error(y_train, model.predict(Xtr)))
    val_err.append(mean_squared_error(y_val,   model.predict(Xval)))

best = alphas[int(np.argmin(val_err))]
plt.figure()
plt.plot(alphas, train_err, color=C_DATA, marker="o", ms=3, label="training error")
plt.plot(alphas, val_err,   color=C_L2,  marker="o", ms=3, label="validation error")
plt.axvline(best, color=C_WARN, ls="--", label=f"best α ≈ {best:.2g}")
plt.xscale("log"); plt.xlabel("regularization α"); plt.ylabel("mean squared error")
plt.title("The train–validation gap closes, then underfitting sets in")
plt.legend(); plt.show()

print(f"Rule of thumb: big train–val gap ⇒ add regularization (or data).")
print(f"Best validation α here ≈ {best:.3g}")

**🔧 Try it.** Set `degree = 3` above and rerun the two cells. A low-capacity model can't overfit this data, so the gap barely appears and `α` matters much less. Capacity and regularization are two knobs on the same trade-off.

## 2 · Weight decay — the `(1 − εα)` shrink

The most common penalty is **half the squared L² norm**:

$$\Omega(\theta)=\tfrac12\lVert w\rVert_2^2=\tfrac12 w^\top w \quad\Longrightarrow\quad \nabla_w\tilde J = \nabla_w J + \alpha w$$

One gradient-descent step (learning rate `ε`) becomes:

$$w \;\leftarrow\; w - \varepsilon(\nabla_w J + \alpha w) \;=\; \underbrace{(1-\varepsilon\alpha)\,w}_{\text{shrink first}} \;-\; \varepsilon\,\nabla_w J$$

Before every ordinary update, the weight is multiplied by a constant **just under 1**. It *decays* — like a slowly leaking balloon. Let's isolate that shrink (set the data gradient aside) and watch a weight that starts at `1.0`.

In [ ]:
def decay_curve(eps, alpha, steps=120, w0=1.0):
    f = 1 - eps * alpha            # per-step shrink factor
    return f, w0 * f ** np.arange(steps)

plt.figure()
for eps, alpha in [(0.1, 0.2), (0.1, 0.5), (0.1, 1.0), (0.05, 1.0)]:
    f, w = decay_curve(eps, alpha)
    plt.plot(w, label=f"ε={eps}, α={alpha}  →  factor {f:.3f}")
plt.xlabel("gradient step"); plt.ylabel("weight value")
plt.title("Weight decay: multiply by (1 − εα) every step")
plt.legend(); plt.show()

print("Notice: the *effective* decay depends on BOTH ε and α — they are entangled.")
print("That coupling is exactly why AdamW decouples weight decay from the learning rate.")

**🔧 Try it — interactive.** Drag the sliders. `factor = 1 − εα`; if `εα > 1` the factor goes negative and the weight *oscillates* — a sign your learning rate is too big. (If sliders don't render, the static plot above already tells the story.)

In [ ]:
# ipywidgets ships with Colab. If it fails, skip this cell — it is optional.
try:
    from ipywidgets import interact, FloatSlider
    def show(eps=0.1, alpha=0.5):
        f, w = decay_curve(eps, alpha, steps=100)
        plt.figure(figsize=(7, 3.6))
        plt.plot(w, color=C_L2, lw=2)
        plt.axhline(0, color="gray", lw=0.8)
        plt.title(f"shrink factor (1 − εα) = {f:.3f}   |   per-step decay = {100*(1-f):.1f}%")
        plt.xlabel("step"); plt.ylabel("weight"); plt.ylim(-1.1, 1.1); plt.grid(alpha=0.25)
        plt.show()
    interact(show,
             eps=FloatSlider(min=0.01, max=0.5, step=0.01, value=0.1, description="ε"),
             alpha=FloatSlider(min=0.0, max=3.0, step=0.05, value=0.5, description="α"));
except Exception as e:
    print("Widgets unavailable here — that's fine, the static plot covers it.", e)

## 3 · Weight decay in a real training loop (PyTorch)

`weight_decay=α` is a one-line argument in every optimizer. Below we train the **same** small network twice — once with `weight_decay=0`, once with a healthy value — on a noisy classification problem, and compare the weight norm and the validation accuracy.

In [ ]:
import torch, torch.nn as nn
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

torch.manual_seed(0)
Xm, ym = make_moons(n_samples=200, noise=0.30, random_state=0)
Xt, Xv, yt, yv = train_test_split(Xm, ym, test_size=0.5, random_state=0)
to_t = lambda a, d=torch.float32: torch.tensor(a, dtype=d)
Xt, Xv = to_t(Xt), to_t(Xv)
yt, yv = to_t(yt), to_t(yv)

def make_net():
    torch.manual_seed(1)
    return nn.Sequential(nn.Linear(2, 64), nn.ReLU(),
                         nn.Linear(64, 64), nn.ReLU(),
                         nn.Linear(64, 1))

def train(weight_decay, epochs=400):
    net = make_net()
    opt = torch.optim.SGD(net.parameters(), lr=0.1, weight_decay=weight_decay)
    lossf = nn.BCEWithLogitsLoss()
    hist = {"val_acc": [], "wnorm": []}
    for _ in range(epochs):
        opt.zero_grad()
        loss = lossf(net(Xt).squeeze(), yt)
        loss.backward(); opt.step()
        with torch.no_grad():
            acc = ((net(Xv).squeeze() > 0).float() == yv).float().mean().item()
            wn = sum(p.pow(2).sum() for n, p in net.named_parameters() if "weight" in n).sqrt().item()
        hist["val_acc"].append(acc); hist["wnorm"].append(wn)
    return hist

h0  = train(weight_decay=0.0)
hwd = train(weight_decay=5e-3)

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(h0["wnorm"],  color=C_WARN, label="weight_decay = 0")
ax[0].plot(hwd["wnorm"], color=C_L2,   label="weight_decay = 5e-3")
ax[0].set_title("Total weight norm ‖w‖"); ax[0].set_xlabel("epoch"); ax[0].legend()
ax[1].plot(h0["val_acc"],  color=C_WARN, label="weight_decay = 0")
ax[1].plot(hwd["val_acc"], color=C_L2,   label="weight_decay = 5e-3")
ax[1].set_title("Validation accuracy"); ax[1].set_xlabel("epoch"); ax[1].legend()
plt.tight_layout(); plt.show()

print(f"final ‖w‖   — no decay: {h0['wnorm'][-1]:.1f}   with decay: {hwd['wnorm'][-1]:.1f}")
print(f"final valacc — no decay: {h0['val_acc'][-1]:.3f}  with decay: {hwd['val_acc'][-1]:.3f}")

**✏️ Exercise 3.** The `weight_decay` value above (`5e-3`) was picked by hand. Loop over `for wd in [0, 1e-4, 1e-3, 5e-3, 1e-2, 5e-2]:`, train each, and record the **final validation accuracy**. Plot accuracy vs `wd` on a log-x axis and read off the best value. You should see the same "too little / too much" U-shape from Part 1.

In [ ]:
# ✏️ Your code here.
# Hint: reuse train(); collect final val_acc; plt.semilogx(wds, accs, marker='o').


## 4 · The Hessian view — which directions survive?

To see *what* L² keeps and what it throws away, approximate the loss near its optimum `w*` by a quadratic bowl whose curvature is the **Hessian** `H`:

$$\hat J(w)=J(w^*)+\tfrac12 (w-w^*)^\top H\,(w-w^*)\quad\Longrightarrow\quad \tilde w=(H+\alpha I)^{-1}H\,w^*$$

In the eigenbasis of `H` each direction `i` (curvature `λᵢ`) is simply **rescaled**:

$$\tilde w_i = \frac{\lambda_i}{\lambda_i+\alpha}\; w^*_i$$

* `λ ≫ α` → factor ≈ **1**: steep directions matter, so they're kept.
* `λ ≪ α` → factor ≈ **0**: flat directions barely affect the loss, so they decay away.
* `λ = α` → factor = **½**: that direction is cut exactly in half.

Let's build an explicit 2-D bowl and watch `w̃` slide from `w*` toward the origin as `α` grows.

In [ ]:
# A 2-D quadratic bowl with a STEEP axis and a FLAT axis (diagonal Hessian for clarity).
lam = np.array([4.0, 0.4])      # curvatures: steep (4.0) and flat (0.4)
H = np.diag(lam)
w_star = np.array([1.0, 1.0])   # unregularized optimum

def w_tilde(alpha):
    return np.linalg.solve(H + alpha * np.eye(2), H @ w_star)

# Contours of J_hat around w*
g = np.linspace(-0.4, 1.4, 240)
A, B = np.meshgrid(g, g)
D0, D1 = A - w_star[0], B - w_star[1]
Z = 0.5 * (lam[0] * D0**2 + lam[1] * D1**2)

plt.figure(figsize=(6.2, 6))
plt.contour(A, B, Z, levels=15, colors=C_DATA, alpha=0.5, linewidths=0.8)
plt.scatter(*w_star, color=C_DATA, s=70, zorder=5, label="w*  (data optimum)")
plt.scatter(0, 0, color="k", s=40, zorder=5, label="origin")

path = np.array([w_tilde(a) for a in np.linspace(0, 40, 60)])
plt.plot(path[:, 0], path[:, 1], color=C_L2, lw=2, label="w̃(α) as α grows")
for a in [0.4, 4.0, 20.0]:
    wt = w_tilde(a); plt.scatter(*wt, color=C_L2, s=45, zorder=6)
    plt.annotate(f"α={a}", wt, textcoords="offset points", xytext=(6, 6), fontsize=9)

plt.gca().set_aspect("equal"); plt.xlabel("w₁ (steep, λ=4.0)"); plt.ylabel("w₂ (flat, λ=0.4)")
plt.title("L² pulls w* toward 0 — but faster along the flat axis")
plt.legend(loc="lower right"); plt.show()

print("Note the path bends toward the flat axis: the flat coordinate (w₂) collapses first.")

**🔧 Try it — the eigen-rescale meter.** Each bar is the surviving fraction `λ/(λ+α)` for one eigen-direction. Drag `α` and watch the flat directions collapse first.

In [ ]:
def rescale_bars(alpha, lambdas=np.array([0.1, 0.4, 1.0, 4.0, 12.0])):
    frac = lambdas / (lambdas + alpha)
    plt.figure(figsize=(7, 3.4))
    colors = [C_L2 if f > 0.5 else C_WARN for f in frac]
    plt.barh([f"λ={l:g}" for l in lambdas], frac, color=colors)
    plt.axvline(0.5, color="gray", ls="--", lw=1)
    plt.xlim(0, 1); plt.xlabel("surviving fraction  λ/(λ+α)")
    plt.title(f"α = {alpha:.2f}   (bars past 0.5 = kept, before 0.5 = decaying)")
    plt.gca().invert_yaxis(); plt.grid(alpha=0.25, axis="x"); plt.show()

try:
    from ipywidgets import interact, FloatSlider
    interact(rescale_bars, alpha=FloatSlider(min=0.0, max=12.0, step=0.1, value=1.0, description="α"));
except Exception:
    for a in [0.0, 1.0, 4.0]:
        rescale_bars(a)

## 5 · Ridge regression — the same story in the data

For linear regression the normal equations pick up an `αI` on the diagonal:

$$\text{ordinary: } w=(X^\top X)^{-1}X^\top y \qquad\longrightarrow\qquad \text{ridge: } w=(X^\top X+\alpha I)^{-1}X^\top y$$

`XᵀX` is proportional to the input covariance, so adding `αI` inflates every feature's apparent variance. High-variance directions (that also covary with the target) are barely moved; low-variance directions are shrunk hardest — the exact same "keep steep, decay flat" rule. Let's watch the **coefficient paths** collapse toward zero as `α` sweeps.

In [ ]:
# Correlated features -> plain least squares is unstable; ridge tames it.
rng = np.random.default_rng(3)
n, d = 60, 8
base = rng.normal(size=(n, 3))
# Build 8 features as noisy mixtures of only 3 latent signals -> strong collinearity
mix = rng.normal(size=(3, d))
X = base @ mix + 0.05 * rng.normal(size=(n, d))
true_w = np.array([3, -2, 0, 0, 1.5, 0, 0, -1.0])
y = X @ true_w + rng.normal(0, 0.5, n)

alphas = np.logspace(-3, 3, 60)
paths = []
for a in alphas:
    w = np.linalg.solve(X.T @ X + a * np.eye(d), X.T @ y)
    paths.append(w)
paths = np.array(paths)

plt.figure()
for j in range(d):
    plt.plot(alphas, paths[:, j], label=f"w{j}")
plt.xscale("log"); plt.xlabel("regularization α"); plt.ylabel("coefficient value")
plt.title("Ridge coefficient paths — every weight is pulled smoothly toward 0")
plt.legend(ncol=2, fontsize=8); plt.show()

# Sanity check against scikit-learn
from sklearn.linear_model import Ridge
skw = Ridge(alpha=10.0, fit_intercept=False).fit(X, y).coef_
myw = np.linalg.solve(X.T @ X + 10.0 * np.eye(d), X.T @ y)
print("closed-form vs sklearn (α=10) max diff:", np.max(np.abs(skw - myw)))

**✏️ Exercise 5.** Notice that in ridge **no coefficient becomes exactly zero** — they only shrink. Repeat this experiment with `sklearn.linear_model.Lasso` (L¹) instead of `Ridge` and plot its paths. You'll see several coefficients hit *exactly* zero. That qualitative jump — shrinkage vs sparsity — is the whole subject of **Notebook 2**.

In [ ]:
# ✏️ Your code here (preview of Notebook 2).
# from sklearn.linear_model import Lasso
# paths_l1 = [Lasso(alpha=a, fit_intercept=False, max_iter=5000).fit(X, y).coef_ for a in alphas]


## Key takeaways

1. **Penalty, not punishment.** `J̃ = J + αΩ` trades a little training error for a smaller train–validation gap. `α` is tuned on validation; we penalize weights, not biases.
2. **L² is a per-step shrink.** Weight decay multiplies `w` by `(1 − εα)` before each step; its effective strength is entangled with the learning rate (hence AdamW).
3. **Curvature decides survival.** `w̃ = (H + αI)⁻¹Hw*`; each direction scales by `λ/(λ+α)`. Steep directions are kept, flat ones decay away.
4. **Ridge is this in the data.** `(XᵀX + αI)⁻¹Xᵀy` shrinks low-variance directions hardest and stabilizes correlated features.

**Next → Notebook 2:** swap the squared norm for the sum of absolute values and shrinkage turns into **sparsity** — soft thresholding, LASSO feature selection, and the Gaussian-vs-Laplace prior behind it all.